# Customer Support Intelligence Platform — Phase 4: Advanced Classical ML & Ensemble Models

**Brief requirements covered (Day 9-10):**
- Train Random Forest, XGBoost, LightGBM on **combined tabular + TF-IDF features**
  (upgrade from baselines, which used only one feature type per task)
- Cross-validation for robust estimation
- Hyperparameter tuning via Optuna
- Log every experiment to MLflow with full parameter and metric tracking

**Expectation set from baseline results:** our shuffle-test confirmed only a
weak (though genuine) signal exists in this dataset. Ensemble methods may
extract more of that weak signal than linear/simple models did, but a
dramatic jump to high accuracy would be surprising given what we've already
measured - documenting whatever we find honestly, not chasing a specific number.

In [2]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import xgboost as xgb
import lightgbm as lgb
import optuna

RANDOM_STATE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)  # keep notebook output readable

train_df = pd.read_csv("../data_processed/train.csv")
val_df = pd.read_csv("../data_processed/val.csv")
test_df = pd.read_csv("../data_processed/test.csv")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 5928, Val: 1270, Test: 1271


## 1. Build combined tabular + TF-IDF feature matrix

Refitting TF-IDF on train only again (same leakage-avoidance discipline as
Day 7-8), then horizontally stacking it with the tabular features into one
combined sparse matrix per row.

In [3]:
TABULAR_FEATURES = [
    "Customer Age", "Ticket Channel_Encoded", "Product Purchased_Encoded",
    "Customer Gender_Encoded", "Customer_Tenure_Days",
    "Description_Char_Count", "Description_Word_Count", "Sentiment_Polarity",
]

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["Combined_Text_Clean"])
X_val_tfidf = tfidf.transform(val_df["Combined_Text_Clean"])
X_test_tfidf = tfidf.transform(test_df["Combined_Text_Clean"])

joblib.dump(tfidf, "../models/tfidf_vectorizer_ensemble.joblib")

def build_combined_features(df, tfidf_matrix):
    tabular = csr_matrix(df[TABULAR_FEATURES].values)
    return hstack([tfidf_matrix, tabular]).tocsr()

X_train_combined = build_combined_features(train_df, X_train_tfidf)
X_val_combined = build_combined_features(val_df, X_val_tfidf)
X_test_combined = build_combined_features(test_df, X_test_tfidf)

print(f"Combined feature matrix shape - train: {X_train_combined.shape}")
print(f"(3000 TF-IDF features + {len(TABULAR_FEATURES)} tabular features)")

y_train_type = train_df["Ticket Type"]
y_val_type = val_df["Ticket Type"]
y_train_priority = train_df["Ticket Priority"]
y_val_priority = val_df["Ticket Priority"]

Combined feature matrix shape - train: (5928, 3008)
(3000 TF-IDF features + 8 tabular features)


In [4]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("customer_support_ensemble_models")

def evaluate_classifier(y_true, y_pred, y_proba, label_names):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", labels=label_names)
    except ValueError as e:
        roc_auc = None
        print(f"  (ROC-AUC computation failed: {e})")
    return acc, f1_macro, roc_auc

## 2. Random Forest — Ticket Type (with cross-validation)

5-fold stratified cross-validation on the training set, per the brief's
"Use cross-validation for robust estimation" instruction.

In [5]:
with mlflow.start_run(run_name="random_forest_ticket_type"):
    rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(rf, X_train_combined, y_train_type, cv=cv, scoring="f1_macro")
    print(f"5-fold CV F1-macro scores: {cv_scores}")
    print(f"Mean CV F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    rf.fit(X_train_combined, y_train_type)
    y_pred = rf.predict(X_val_combined)
    y_proba = rf.predict_proba(X_val_combined)
    acc_rf_type, f1_rf_type, roc_rf_type = evaluate_classifier(y_val_type, y_pred, y_proba, sorted(y_train_type.unique()))

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_metric("cv_f1_macro_mean", cv_scores.mean())
    mlflow.log_metric("cv_f1_macro_std", cv_scores.std())
    mlflow.log_metric("accuracy", acc_rf_type)
    mlflow.log_metric("f1_macro", f1_rf_type)
    if roc_rf_type:
        mlflow.log_metric("roc_auc", roc_rf_type)
    mlflow.sklearn.log_model(rf, "model")

    print(f"\nRandom Forest - Ticket Type (validation)")
    print(f"  Accuracy: {acc_rf_type:.4f}  F1-macro: {f1_rf_type:.4f}  ROC-AUC: {roc_rf_type:.4f}")

5-fold CV F1-macro scores: [0.18508676 0.18446246 0.15912199 0.17913163 0.17746985]
Mean CV F1-macro: 0.1771 (+/- 0.0094)


2026/09/10 14:39:46 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



Random Forest - Ticket Type (validation)
  Accuracy: 0.1850  F1-macro: 0.1703  ROC-AUC: 0.4933


## 3. XGBoost — Ticket Type, with Optuna hyperparameter tuning

Optuna searches for good hyperparameters by trying different combinations
and learning from each trial's result - more efficient than a manual grid
search, per the brief's explicit Optuna requirement.

In [6]:
# XGBoost needs integer-encoded labels, not strings
type_label_map = {label: i for i, label in enumerate(sorted(y_train_type.unique()))}
type_label_map_inv = {v: k for k, v in type_label_map.items()}
y_train_type_enc = y_train_type.map(type_label_map)
y_val_type_enc = y_val_type.map(type_label_map)

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": RANDOM_STATE,
        "eval_metric": "mlogloss",
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_combined, y_train_type_enc)
    preds = model.predict(X_val_combined)
    return f1_score(y_val_type_enc, preds, average="macro")

print("Running Optuna search (20 trials)...")
study = optuna.create_study(direction="maximize")
study.optimize(xgb_objective, n_trials=20, show_progress_bar=True)

print(f"\nBest F1-macro from search: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

Running Optuna search (20 trials)...


Best trial: 2. Best value: 0.209578: 100%|██████████| 20/20 [16:38<00:00, 49.91s/it]


Best F1-macro from search: 0.2096
Best params: {'n_estimators': 261, 'max_depth': 10, 'learning_rate': 0.011719224446838442, 'subsample': 0.7540781138890367, 'colsample_bytree': 0.7247256720462841}


In [7]:
with mlflow.start_run(run_name="xgboost_ticket_type_optuna_tuned"):
    best_xgb = xgb.XGBClassifier(**study.best_params, random_state=RANDOM_STATE, eval_metric="mlogloss")
    best_xgb.fit(X_train_combined, y_train_type_enc)

    y_pred_enc = best_xgb.predict(X_val_combined)
    y_proba = best_xgb.predict_proba(X_val_combined)
    y_pred = pd.Series(y_pred_enc).map(type_label_map_inv)

    acc_xgb_type, f1_xgb_type, roc_xgb_type = evaluate_classifier(y_val_type_enc, y_pred_enc, y_proba, list(range(len(type_label_map))))

    for param, value in study.best_params.items():
        mlflow.log_param(param, value)
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("tuning_method", "Optuna (20 trials)")
    mlflow.log_metric("accuracy", acc_xgb_type)
    mlflow.log_metric("f1_macro", f1_xgb_type)
    if roc_xgb_type:
        mlflow.log_metric("roc_auc", roc_xgb_type)
    mlflow.sklearn.log_model(best_xgb, "model")

    print(f"XGBoost (Optuna-tuned) - Ticket Type (validation)")
    print(f"  Accuracy: {acc_xgb_type:.4f}  F1-macro: {f1_xgb_type:.4f}  ROC-AUC: {roc_xgb_type:.4f}")

2026/09/10 14:57:50 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


XGBoost (Optuna-tuned) - Ticket Type (validation)
  Accuracy: 0.2102  F1-macro: 0.2096  ROC-AUC: 0.4909


## 4. LightGBM — Ticket Priority (tabular + TF-IDF combined)

In [8]:
priority_label_map = {label: i for i, label in enumerate(sorted(y_train_priority.unique()))}
priority_label_map_inv = {v: k for k, v in priority_label_map.items()}
y_train_priority_enc = y_train_priority.map(priority_label_map)
y_val_priority_enc = y_val_priority.map(priority_label_map)

# Cross-validation (matching the treatment Random Forest got)
cv_lgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
base_lgb = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
cv_scores_lgb = cross_val_score(base_lgb, X_train_combined, y_train_priority_enc, cv=cv_lgb, scoring="f1_macro")
print(f"LightGBM 5-fold CV F1-macro scores: {cv_scores_lgb}")
print(f"Mean CV F1-macro: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})")

# Optuna hyperparameter tuning (matching the treatment XGBoost got)
def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "random_state": RANDOM_STATE,
        "verbose": -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_combined, y_train_priority_enc)
    preds = model.predict(X_val_combined)
    return f1_score(y_val_priority_enc, preds, average="macro")

print("\nRunning Optuna search for LightGBM (20 trials)...")
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=20, show_progress_bar=True)
print(f"Best F1-macro from search: {study_lgb.best_value:.4f}")
print(f"Best params: {study_lgb.best_params}")

with mlflow.start_run(run_name="lightgbm_ticket_priority_optuna_tuned"):
    lgb_model = lgb.LGBMClassifier(**study_lgb.best_params, random_state=RANDOM_STATE, verbose=-1)
    lgb_model.fit(X_train_combined, y_train_priority_enc)

    y_pred_enc = lgb_model.predict(X_val_combined)
    y_proba = lgb_model.predict_proba(X_val_combined)
    y_pred = pd.Series(y_pred_enc).map(priority_label_map_inv)

    acc_lgb_priority, f1_lgb_priority, roc_lgb_priority = evaluate_classifier(
        y_val_priority_enc, y_pred_enc, y_proba, list(range(len(priority_label_map)))
    )

    for param, value in study_lgb.best_params.items():
        mlflow.log_param(param, value)
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("tuning_method", "Optuna (20 trials)")
    mlflow.log_metric("cv_f1_macro_mean", cv_scores_lgb.mean())
    mlflow.log_metric("cv_f1_macro_std", cv_scores_lgb.std())
    mlflow.log_metric("accuracy", acc_lgb_priority)
    mlflow.log_metric("f1_macro", f1_lgb_priority)
    if roc_lgb_priority:
        mlflow.log_metric("roc_auc", roc_lgb_priority)
    mlflow.sklearn.log_model(lgb_model, "model")

    print(f"\nLightGBM (Optuna-tuned) - Ticket Priority (validation)")
    print(f"  Accuracy: {acc_lgb_priority:.4f}  F1-macro: {f1_lgb_priority:.4f}  ROC-AUC: {roc_lgb_priority:.4f}")

c:\Users\CHARU\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\CHARU\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\CHARU\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\CHARU\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  

LightGBM 5-fold CV F1-macro scores: [0.25701487 0.27183733 0.23923027 0.27464275 0.26471997]
Mean CV F1-macro: 0.2615 (+/- 0.0127)

Running Optuna search for LightGBM (20 trials)...


Best trial: 14. Best value: 0.276462: 100%|██████████| 20/20 [02:07<00:00,  6.38s/it]


Best F1-macro from search: 0.2765
Best params: {'n_estimators': 217, 'max_depth': 6, 'learning_rate': 0.12622788897753512, 'num_leaves': 94}


2026/09/10 15:00:38 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



LightGBM (Optuna-tuned) - Ticket Priority (validation)
  Accuracy: 0.2772  F1-macro: 0.2765  ROC-AUC: 0.5204


## 5. Comparison against baselines

Did combining features + ensembles + tuning actually help, or does the
weak-signal finding hold regardless of model sophistication?

In [9]:
print("COMPARISON: Baseline vs Advanced Models")
print("=" * 60)
print(f"{'Model':<35s} {'Task':<20s} {'Accuracy':>10s}")
print("-" * 65)
print(f"{'Logistic Regression (baseline)':<35s} {'Ticket Type':<20s} {0.2071:>10.4f}")
print(f"{'Naive Bayes (baseline)':<35s} {'Ticket Type':<20s} {0.2134:>10.4f}")
print(f"{'Random Forest (tabular+TFIDF)':<35s} {'Ticket Type':<20s} {acc_rf_type:>10.4f}")
print(f"{'XGBoost (Optuna-tuned)':<35s} {'Ticket Type':<20s} {acc_xgb_type:>10.4f}")
print()
print(f"{'Decision Tree (baseline)':<35s} {'Ticket Priority':<20s} {0.2717:>10.4f}")
print(f"{'LightGBM (Optuna-tuned)':<35s} {'Ticket Priority':<20s} {acc_lgb_priority:>10.4f}")
print()
print("If these numbers are close to baseline, that CONFIRMS the weak-signal")
print("finding holds regardless of model sophistication - a legitimate, honest")
print("conclusion about this specific dataset, not a failure of these models.")

COMPARISON: Baseline vs Advanced Models
Model                               Task                   Accuracy
-----------------------------------------------------------------
Logistic Regression (baseline)      Ticket Type              0.2071
Naive Bayes (baseline)              Ticket Type              0.2134
Random Forest (tabular+TFIDF)       Ticket Type              0.1850
XGBoost (Optuna-tuned)              Ticket Type              0.2102

Decision Tree (baseline)            Ticket Priority          0.2717
LightGBM (Optuna-tuned)             Ticket Priority          0.2772

If these numbers are close to baseline, that CONFIRMS the weak-signal
finding holds regardless of model sophistication - a legitimate, honest
conclusion about this specific dataset, not a failure of these models.
